In [ ]:
import csv
import re
import os
import sys

# === CONFIGURATION ===
ORIGINAL_RULES_FILE = "network_rules.txt"    # Original BooleanNet file
PRUNED_EDGES_FILE = "deleted_links.csv"          # Output from the previous script
PRUNED_RULES_FILE = "rules_pruned.txt"            # Final output for BooleanNet

def load_edges_to_remove(file_path):
    """
    Constructs a dictionary mapping: Target -> List of Genes to remove (Sources).
    Parses the CSV file generated by the upstream script containing columns:
    Target_Gene, Source_Gene, Interaction_Type, Reason.
    """
    removals = {}
    try:
        with open(file_path, "r", encoding='utf-8-sig') as f:
            # Automated or enforced delimiter detection
            sample = f.read(1024)
            f.seek(0)
            sniffer = csv.Sniffer()
            try:
                delimiter = sniffer.sniff(sample).delimiter
            except:
                delimiter = ";" # Consistent fallback based on the previous script
            
            reader = csv.DictReader(f, delimiter=delimiter)
            
            # Clean column names (removal of trailing whitespace)
            reader.fieldnames = [x.strip() for x in reader.fieldnames]
            
            # Verify the presence of required headers
            if "Target_Gene" not in reader.fieldnames or "Source_Gene" not in reader.fieldnames:
                print(f"ERROR: The file {file_path} lacks 'Target_Gene' and 'Source_Gene' headers.")
                print(f"Identified headers: {reader.fieldnames}")
                sys.exit()

            for row in reader:
                # --- CRITICAL MODIFICATION ---
                # Reading the correct columns generated by the upstream script
                target = row["Target_Gene"].strip()  # The gene containing the rule to modify
                source = row["Source_Gene"].strip()  # The upstream regulatory gene to be excised
                
                if target not in removals:
                    removals[target] = []
                removals[target].append(source)
                
    except Exception as e:
        print(f"Error parsing CSV file {file_path}: {e}")
        sys.exit()
    return removals

def apply_rule_surgery(rule, sources_to_remove):
    """
    Substitutes occurrences of the source genes with 'False' within the rule string.
    """
    new_rule = rule
    for source in sources_to_remove:
        # Regex: \b denotes a word boundary. Matches 'source' only as an exact full word.
        # Example: When removing "AKT", "AKT1" or "pAKT" must remain intact.
        pattern = r'\b' + re.escape(source) + r'\b'
        
        # Substitute the gene name with "False".
        # Within the BooleanNet framework:
        # - "A or B" without B becomes "A or False" -> logically equivalent to A.
        # - "A and B" without B becomes "A and False" -> disables the activation (correct if B was essential).
        # - "not B" without B becomes "not False" -> True (correct, removal of inhibition = activation).
        new_rule = re.sub(pattern, 'False', new_rule)
        
    return new_rule

def main():
    print("--- INITIATING BOOLEANNET RULE PRUNING ---")
    
    if not os.path.exists(PRUNED_EDGES_FILE):
        print(f"Error: The file {PRUNED_EDGES_FILE} does not exist. Please execute the pruning script first.")
        return

    # 1. Load the removal mapping
    removal_map = load_edges_to_remove(PRUNED_EDGES_FILE)
    print(f"Identified targets for removal across {len(removal_map)} target nodes.")

    # 2. Parse the original rules file and generate the pruned output
    modification_count = 0
    affected_genes_count = 0
    
    try:
        with open(ORIGINAL_RULES_FILE, "r", encoding="utf-8") as f_in, \
             open(PRUNED_RULES_FILE, "w", encoding="utf-8") as f_out:
            
            for line in f_in:
                line_clean = line.strip()
                
                # Skip blank lines or comments
                if not line_clean or line_clean.startswith("#"):
                    f_out.write(line)
                    continue
                
                # Parse the BooleanNet rule format: "Target *= Expression"
                if "*=" in line_clean:
                    parts = line_clean.split("*=", 1)
                    target = parts[0].strip()
                    expression = parts[1].strip()
                    
                    # If this target possesses edges designated for elimination
                    if target in removal_map:
                        sources = removal_map[target]
                        
                        # Apply rule excision
                        new_expression = apply_rule_surgery(expression, sources)
                        
                        # Write the modified rule record
                        f_out.write(f"{target} *= {new_expression}\n")
                        
                        # Metrics computation
                        if new_expression != expression:
                            affected_genes_count += 1
                            # Estimate the number of genes removed within this specific rule
                            # (Approximation based on the source list length)
                            modification_count += len(sources)
                    else:
                        # Retain the original rule if no modifications are required
                        f_out.write(line)
                else:
                    # Non-standard format or configuration line; replicated exactly
                    f_out.write(line)
                    
    except Exception as e:
        print(f"Error during the processing of the rules file: {e}")
        return

    print("\n--- OPERATION SUCCESSFULLY COMPLETED ---")
    print(f"Modified target nodes: {affected_genes_count}")
    print(f"Total regulatory interactions removed (substituted with False): {modification_count}")
    print(f"Generated pruned rules file: {PRUNED_RULES_FILE}")

if __name__ == "__main__":
    main()